#**Titanic Survival Prediction using Machine Learning**

## Objective

The goal of this project is to analyze the Titanic dataset and build a machine learning model to predict whether a passenger survived or not.

The project focuses on understanding key factors that influenced survival and applying basic machine learning techniques to make predictions.

In [37]:
import pandas as pd
df = pd.read_csv('train.csv')
df.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [38]:
df.shape

(891, 12)

In [39]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


## Feature Engineering

A new feature called **FamilySize** was created using:

FamilySize = SibSp + Parch + 1

This feature represents the total number of family members traveling together, including the passenger.

It helps capture social structure, which may influence survival probability.

In [40]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FamilySize
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,2
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,2
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,2
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,1


## Data Preprocessing

- Selected relevant features: Pclass, Sex, Age, Fare, and FamilySize
- Handled missing values in the Age column using the median
- Converted categorical variables (Sex) into numerical format using one-hot encoding

In [41]:
features = ['Pclass', 'Sex', 'Age', 'Fare', 'FamilySize']
X = df[features].copy()
y = df['Survived']

In [42]:
X['Age'] = X['Age'].fillna(X['Age'].median())

In [43]:
X = pd.get_dummies(X)

## Model Training

A Logistic Regression model was used to predict survival.

The data was split into training and validation sets to evaluate how well the model generalizes to unseen data.
The model was trained on selected features and used to predict survival on unseen data.
Logistic Regression was chosen as it is a simple and interpretable model suitable for binary classification problems.

In [44]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)

model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=2000)

## Model Evaluation

The model achieved an accuracy of approximately **77%** on the validation set.

This indicates that the model can reasonably predict passenger survival using the selected features.

In [45]:
from sklearn.metrics import accuracy_score
preds = model.predict(X_val)
accuracy = accuracy_score(y_val, preds)
print(accuracy)

0.8156424581005587


In [46]:
list(X.columns)

['Pclass', 'Age', 'Fare', 'FamilySize', 'Sex_female', 'Sex_male']

### Interpretation of Coefficients

- Features with positive coefficients increase survival probability
- Features with negative coefficients decrease survival probability

For example:
- Being female (Sex) has a strong positive impact on survival
- Higher class and fare also positively influence survival

In [47]:
model.coef_

array([[-0.96827615, -0.03023314,  0.00345832, -0.23369652,  1.31404613,
        -1.31421803]])

In [48]:
df.groupby('Sex')['Survived'].mean()

,Survived
Sex,
female,0.742038
male,0.188908


In [49]:
df.groupby('Pclass')['Survived'].mean()

,Survived
Pclass,
1,0.629630
2,0.472826
3,0.242363


In [53]:
df['AgeGroup'] = pd.cut(df['Age'], bins=5)
df.groupby('AgeGroup')['Survived'].mean()

/tmp/ipykernel_1891/3009823798.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('AgeGroup')['Survived'].mean()


,Survived
AgeGroup,
"(0.34, 16.336]",0.550000
"(16.336, 32.252]",0.369942
"(32.252, 48.168]",0.404255
"(48.168, 64.084]",0.434783
"(64.084, 80.0]",0.090909


In [51]:
df.groupby('Fare')['Survived'].mean()

,Survived
Fare,
0.0000,0.066667
4.0125,0.000000
5.0000,0.000000
6.2375,0.000000
6.4375,0.000000
...,...
227.5250,0.750000
247.5208,0.500000
262.3750,1.000000


In [52]:
df.groupby('FamilySize')['Survived'].mean()

,Survived
FamilySize,
1,0.303538
2,0.552795
3,0.578431
4,0.724138
5,0.200000
6,0.136364
7,0.333333
8,0.000000
11,0.000000


## Key Insights

- Female passengers had a significantly higher survival rate compared to male passengers.
- Passengers in higher classes (Pclass 1) had better survival chances than those in lower classes.
- Higher fare values were associated with higher survival, likely because fare is linked to passenger class.
- Family size showed an interesting pattern, where passengers with moderate family sizes had better survival rates compared to those traveling alone or in very large families.
- Age did not show a strong linear relationship with survival, indicating other factors had a greater impact.

## Conclusion

This project demonstrates how basic data analysis and machine learning techniques can be used to extract meaningful insights and make predictions.

Logistic Regression provided a simple yet effective model for understanding survival patterns in the Titanic dataset.
